# Shared feature and evaluation functions
Used by all XGBoost notebooks. Raw data, fixed folds and outputs stay inside `models`.

In [ ]:
import ctypes  # Load the bundled OpenMP runtime.
ctypes.CDLL(str(ROOT / 'environment/lib/libomp.dylib'))  # Use the packaged library.
import csv, gzip, json, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


## MAP@200 and original features
Duplicate user–event pairs use maximum label and prediction. Users receive equal weight. Categories are fitted separately within each training fold.

### Locate a raw table


In [ ]:
def locate(root, name):
    return next(p for p in [root / name, root / (name + '.gz')] if p.exists())



### Load selected social relationships


In [ ]:
def relationships(path, key, wanted, columns):
    result = {}
    opener = gzip.open if path.suffix == '.gz' else open
    with opener(path, 'rt', newline='') as stream:
        for row in csv.DictReader(stream):
            if row[key] in wanted:
                result[row[key]] = {c: set(row[c].split()) for c in columns}
    return result



### Compute user-level MAP@200


In [ ]:
def map_users(frame, scores):
    ranked = frame[['user', 'event', 'interested']].assign(score=scores)
    ranked = ranked.groupby(['user', 'event'], as_index=False).agg(interested=('interested', 'max'), score=('score', 'max'))  # Deduplicate user-event pairs with maximum label and score.
    ranked = ranked.sort_values(['user', 'score', 'event'], ascending=[True, False, True], kind='stable')  # Break prediction ties deterministically by event ID.
    records = []  # Keep per-user scores for reproducible evaluation.
    for user, group in ranked.groupby('user', sort=False):  # Average users equally, not by candidate count.
        labels = group.interested.to_numpy()[:200]
        positives = int(group.interested.sum())
        ap = float((np.cumsum(labels) / np.arange(1, len(labels) + 1) * labels).sum() / min(positives, 200)) if positives else 0.0
        records.append({'user': user, 'ap200': ap, 'positives': positives})
    return pd.DataFrame(records)



### Build the original 142 features


In [ ]:
def features(root, samples):
    users = pd.read_csv(locate(root, 'users.csv'), dtype=str).set_index('user_id')
    wanted = set(samples.event)
    parts = []
    total = 0
    for chunk in pd.read_csv(locate(root, 'events.csv'), dtype=str, chunksize=50000):
        total += len(chunk)
        parts.append(chunk.loc[chunk.event_id.isin(wanted)])
    events = pd.concat(parts, ignore_index=True).set_index('event_id')
    u = users.reindex(samples.user).reset_index(drop=True)
    e = events.reindex(samples.event).reset_index(drop=True)
    x = pd.DataFrame(index=samples.index)
    x['invited'] = pd.to_numeric(samples.invited, errors='coerce')  # Use invitation status at exposure time.
    observed = pd.to_datetime(samples.timestamp, utc=True, errors='coerce', format='mixed')
    start = pd.to_datetime(e.start_time, utc=True, errors='coerce', format='mixed')
    joined = pd.to_datetime(u.joinedAt, utc=True, errors='coerce', format='mixed')
    x['lead_hours'] = (start - observed).dt.total_seconds() / 3600
    x['tenure_days'] = (observed - joined).dt.total_seconds() / 86400
    birth = pd.to_numeric(u.birthyear, errors='coerce')
    x['age'] = (observed.dt.year - birth).where(birth.between(1900, 2013))
    for name, dates in [('shown', observed), ('start', start)]:
        x[name + '_hour'] = dates.dt.hour
        x[name + '_weekday'] = dates.dt.dayofweek
        x[name + '_month'] = dates.dt.month
        x[name + '_day'] = dates.dt.day
    x['timezone'] = pd.to_numeric(u.timezone, errors='coerce')  # Use the original timezone field.
    for name in ['lat', 'lng']:
        x[name] = pd.to_numeric(e[name], errors='coerce')
    for name in ['locale', 'gender']:
        x['user_' + name] = u[name].fillna('__missing__').str.lower().str.strip()
    location = u.location.fillna('').str.lower().str.strip()
    for name in ['city', 'state', 'country']:
        values = e[name].fillna('').str.lower().str.strip()
        x['event_' + name] = values.replace('', '__missing__')
        x['location_matches_' + name] = [float(v in loc) if v and loc else np.nan for v, loc in zip(values, location)]
    words = e.filter(regex=r'^c_').apply(pd.to_numeric, errors='coerce')
    x['content_sum'] = words.sum(axis=1, min_count=1)
    x['content_nonzero'] = words.gt(0).sum(axis=1).where(words.notna().any(axis=1))
    x = pd.concat([x, np.log1p(words.clip(lower=0)).add_prefix('log_')], axis=1)
    x['user_missing'] = ~samples.user.isin(users.index)
    x['event_missing'] = ~samples.event.isin(events.index)
    x['candidate_count'] = samples.groupby('user').event.transform('nunique')  # Candidate count does not use labels.
    friends = relationships(locate(root, 'user_friends.csv'), 'user', set(samples.user), ['friends'])
    statuses = ['yes', 'maybe', 'invited', 'no']
    attendees = relationships(locate(root, 'event_attendees.csv'), 'event', wanted, statuses)
    rows = []
    for user, event in zip(samples.user, samples.event):
        f = friends.get(user, {}).get('friends')
        f = f - {user} if f is not None else None
        row = {'friend_count': len(f) if f is not None else np.nan}
        for status in statuses:
            a = attendees.get(event, {}).get(status)
            a = a - {user} if a is not None else None  # Exclude the target user from event response statistics.
            row[status + '_count'] = len(a) if a is not None else np.nan
            row[status + '_friends'] = len(a & f) if a is not None and f is not None else np.nan
            row[status + '_friend_fraction'] = row[status + '_friends'] / max(len(f), 1) if f is not None else np.nan
        rows.append(row)
    x = pd.concat([x, pd.DataFrame(rows)], axis=1)
    audit = {'events_source_rows': total, 'events_selected': len(events), 'users_source_rows': len(users), 'user_missing_rows': int(x.user_missing.sum()), 'event_missing_rows': int(x.event_missing.sum()), 'duplicate_pairs': int(samples.duplicated(['user', 'event']).sum())}
    return x, audit


## Experiment 05 cleaning and fold-specific encoding

### Normalize category text


In [ ]:
def normalize(value):
    if pd.isna(value):
        return ''
    return ' '.join(unicodedata.normalize('NFKC', str(value)).casefold().split())



### Apply Experiment 05 cleaning


In [ ]:
def clean(frame, samples):
    x = frame.copy(deep=True)
    audit = dict(rows=len(x), original_columns=len(x.columns), changes={})
    nums = x.select_dtypes(include='number').columns
    for column in nums:  # Let XGBoost handle missing numeric values.
        invalid = np.isinf(x[column].to_numpy())
        if invalid.any():
            audit['changes'][column + '_infinite'] = int(invalid.sum())
            x.loc[invalid, column] = np.nan
    for column, lower, upper in [('age', 0, 120), ('lat', -90, 90), ('lng', -180, 180), ('timezone', -720, 840)]:
        invalid = x[column].notna() & ~x[column].between(lower, upper)
        audit['changes'][column + '_outside_range'] = int(invalid.sum())
        x.loc[invalid, column] = np.nan
    negative = x.tenure_days < 0
    x['tenure_invalid'] = negative.astype(np.float32)
    x.loc[negative, 'tenure_days'] = np.nan
    audit['changes']['negative_tenure'] = int(negative.sum())
    for column in x.select_dtypes(include=['object', 'string']):
        values = x[column].map(normalize).replace('', '__missing__')  # Use a consistent missing-category marker.
        audit['changes'][column + '_normalized'] = int((values != x[column]).sum())
        x[column] = values
    users = pd.read_csv(ROOT / 'data/raw/users.csv', dtype=str).set_index('user_id')
    locations = users.location.reindex(samples.user).reset_index(drop=True).map(normalize)
    for region in ['city', 'state', 'country']:
        values = x['event_' + region]
        pairs = list(zip(values, locations))
        cache = {}
        for value, location in set(pairs):
            cache[(value, location)] = float(re.search(r'(?<!\w)' + re.escape(value) + r'(?!\w)', location) is not None) if value != '__missing__' and location else np.nan  # Match complete geographic phrases rather than arbitrary substrings.
        matched = pd.Series([cache[pair] for pair in pairs], index=x.index)
        column = 'location_matches_' + region
        same = matched.eq(x[column]) | (matched.isna() & x[column].isna())
        audit['changes'][column + '_boundary_corrected'] = int((~same).sum())
        x[column] = matched
    audit.update(final_columns=len(x.columns), rows_preserved=len(x) == len(frame), all_original_columns_preserved=set(frame.columns).issubset(x.columns), negative_lead_hours_retained=int((x.lead_hours < 0).sum()))
    return x, audit



### Fit fold-specific categorical encoding


In [ ]:
def encode(x, fit):
    cats = x.select_dtypes(include=['object', 'string']).columns.tolist()
    nums = [column for column in x if column not in cats]
    encoder = OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False, dtype=np.float32)  # Pool rare categories within each training split, as in the baseline.
    encoder.fit(x.iloc[fit][cats])  # Fit categories on training rows only.
    matrix = np.column_stack([x[nums].to_numpy(dtype=np.float32), encoder.transform(x[cats])])  # Dense encoding preserves the distinction between zero and missing.
    return matrix, encoder, nums + encoder.get_feature_names_out(cats).tolist()


## Friend profiles
For each friend, average event fields across recorded responses. Then average available friends equally. `yes` and `no` select positive and negative histories respectively.

### Construct positive or negative friend profiles


In [ ]:
def friend_profile(samples, status):
    wanted=set(samples.user)
    friends={}
    with gzip.open(ROOT/'data/raw/user_friends.csv.gz','rt') as stream:
     for row in csv.DictReader(stream):
      if row['user'] in wanted: friends[row['user']]=set(row['friends'].split())-{row['user']}
    relevant=set().union(*friends.values())
    relations={}
    with (ROOT/'data/raw/event_attendees.csv').open() as stream:
     for row in csv.DictReader(stream):
      members=relevant.intersection(row[status].split()) # Use response records, not supervised interest labels.
      if members: relations[int(row['event'])]=members
    columns=['start_time','lat','lng']+[f'c_{i}' for i in range(1,101)]+['c_other']
    events=np.array(list(relations),dtype=np.uint64)
    parts=[]
    for chunk in pd.read_csv(ROOT/'data/raw/events.csv.gz',usecols=['event_id']+columns,dtype=str,chunksize=50000):
     parts.append(chunk.loc[chunk.event_id.isin(set(map(str,events)))])
    raw=pd.concat(parts,ignore_index=True).set_index('event_id')
    valid=np.array([str(e) in raw.index for e in events])
    raw=raw.reindex(list(map(str,events)))
    dates=pd.to_datetime(raw.start_time,utc=True,errors='coerce',format='mixed')
    raw['start_time']=dates.map(lambda t:t.timestamp() if pd.notna(t) else np.nan) # Represent time as UTC Unix seconds; retain missing timestamps.
    latent=raw[columns].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64) # Use raw numeric values without logarithms or standardization.
    latent[~np.isfinite(latent)]=np.nan
    selected=np.flatnonzero(valid)
    sums={}
    counts={}
    for i in selected:
     for user in relations[int(events[i])]: # Count each recorded response once.
      if user not in sums: sums[user]=np.zeros(104,dtype=np.float64);counts[user]=np.zeros(104,dtype=np.int64)
      sums[user]+=np.nan_to_num(latent[i],nan=0.0);counts[user]+=np.isfinite(latent[i])
    profiles={u:np.divide(sums[u],counts[u],out=np.full(104,np.nan),where=counts[u]>0) for u in sums} # First average events separately for each friend.
    associated={}
    for user in sorted(wanted):
     available=[profiles[f] for f in friends.get(user,set()) if f in profiles]
     associated[user]=np.divide(np.nansum(available,axis=0),np.isfinite(available).sum(axis=0),out=np.full(104,np.nan),where=np.isfinite(available).sum(axis=0)>0) if available else np.full(104,np.nan) # Average friends equally, rather than weighting by their activity counts.
    return pd.DataFrame(np.stack([associated[u] for u in samples.user]),columns=['friend_raw_'+c for c in columns])


## Fixed-parameter five-fold training and saved-model evaluation

### Evaluate packaged models


In [ ]:
def evaluate_saved(x, train, folds, reference, seeds, baseline=False):
    scores = np.zeros(len(train))
    for fold in range(1, 6):
        valid = np.flatnonzero(folds == fold)
        encoder = pd.read_pickle(reference / (f'encoder_{fold}.pkl' if baseline else f'fold_{fold}_encoder.pkl'))
        cats = x.select_dtypes(include=['object', 'string']).columns.tolist()
        nums = [c for c in x if c not in cats]
        matrix = np.column_stack([x[nums].to_numpy(dtype=np.float32), encoder.transform(x[cats])])
        for seed in seeds:
            model = XGBClassifier()
            model.load_model(reference / (f'fold_{fold}.ubj' if baseline else f'fold_{fold}_seed_{seed}.ubj'))
            scores[valid] += model.predict_proba(matrix[valid])[:, 1] / len(seeds)
    return scores



### Train five folds with fixed parameters


In [ ]:
def train_folds(x, train, test, folds, parameters, seeds, output, weight_directory=None):
    oof, test_scores = np.zeros(len(train)), np.zeros(len(test))
    for fold in range(1, 6):
        fit, valid = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        matrix, encoder, names = encode(x, fit)
        pd.to_pickle(encoder, output / f'fold_{fold}_encoder.pkl')
        for seed in seeds:
            model = XGBClassifier(**dict(parameters, random_state=seed))
            if weight_directory is not None:
                weights = pd.read_csv(weight_directory / f'feature_weights_{fold}.csv').set_index('feature')
                model.set_params(feature_weights=weights.loc[names, 'weight'].to_numpy(dtype=np.float32))
            model.fit(matrix[fit], train.interested.to_numpy()[fit])
            oof[valid] += model.predict_proba(matrix[valid])[:, 1] / len(seeds)
            test_scores += model.predict_proba(matrix[len(train):])[:, 1] / (5 * len(seeds))
            model.save_model(output / f'fold_{fold}_seed_{seed}.ubj')
        print(fold, map_users(train.iloc[valid], oof[valid]).ap200.mean())
    train[['user','event','interested']].assign(fold=folds, probability=oof).to_csv(output / 'oof.csv', index=False)
    ranked = test[['user','event']].assign(probability=test_scores).sort_values(['user','probability','event'], ascending=[True,False,True])
    submission = ranked.drop_duplicates(['user','event']).groupby('user').event.agg(' '.join).reset_index()
    submission.columns = ['User','Events']
    submission.iloc[np.argsort(submission.User.astype('uint64').to_numpy())].to_csv(output / 'submission.csv', index=False)
    score = float(map_users(train, oof).ap200.mean())
    (output / 'metrics.json').write_text(json.dumps({'oof_map200':score,'parameters':parameters,'seeds':seeds},indent=2))
    return score

